In [9]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StandardScaler, VectorAssembler
from pyspark.sql.functions import col, expr, lit
from functools import reduce
import glob
import os

In [10]:
spark=SparkSession.builder.appName('Healthcare_DataCleaning').getOrCreate()

In [11]:
folder_path = "Data/*.csv"
file_paths = glob.glob(folder_path)

In [12]:
sample_df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_paths[0])
print("Schema:")
sample_df.printSchema()
print("\nFirst 5 rows:")
sample_df.show(5)
print("\nColumn stats:")
sample_df.describe().show()

Schema:
root
 |-- Rndrng_Prvdr_Geo_Lvl: string (nullable = true)
 |-- Rndrng_Prvdr_Geo_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_Geo_Desc: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable = true)
 |-- HCPCS_Drug_Ind: string (nullable = true)
 |-- Place_Of_Srvc: string (nullable = true)
 |-- Tot_Rndrng_Prvdrs: integer (nullable = true)
 |-- Tot_Benes: integer (nullable = true)
 |-- Tot_Srvcs: double (nullable = true)
 |-- Tot_Bene_Day_Srvcs: integer (nullable = true)
 |-- Avg_Sbmtd_Chrg: double (nullable = true)
 |-- Avg_Mdcr_Alowd_Amt: double (nullable = true)
 |-- Avg_Mdcr_Pymt_Amt: double (nullable = true)
 |-- Avg_Mdcr_Stdzd_Amt: double (nullable = true)


First 5 rows:
+--------------------+-------------------+---------------------+--------+--------------------+--------------+-------------+-----------------+---------+---------+------------------+--------------+------------------+-----------------+------------------+
|Rndrng_Prv

In [13]:
target_cols = ['Rndrng_Prvdr_Geo_Cd','HCPCS_Cd','HCPCS_Drug_Ind','Place_Of_Srvc', 'Tot_Rndrng_Prvdrs','Tot_Benes','Tot_Srvcs','Tot_Bene_Day_Srvcs','Avg_Sbmtd_Chrg']

In [14]:
def clean_data(df_data):
    return (df_data.filter(col('Rndrng_Prvdr_Geo_Lvl') != 'National')
            .na.drop()
            .filter(~expr("try_cast(Rndrng_Prvdr_Geo_Cd as int)").isNull())
            .select(target_cols))

In [15]:
def get_year(file_path):
    year_part = os.path.basename(file_path).split('.')[0]
    return year_part

In [16]:
all_data = []
for file_path in file_paths:
    year = get_year(file_path)
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    cleaned_df = clean_data(df)
    cleaned_df = cleaned_df.withColumn("Year", lit(year))
    all_data.append(cleaned_df)

{"ts": "2026-04-05 22:55:52.525", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Rndrng_Prvdr_Geo_Lvl` cannot be resolved. Did you mean one of the following? [`Rndrng_Prvdr_Geo_Cd`, `Tot_Rndrng_Prvdrs`, `Avg_Sbmtd_Chrg`, `HCPCS_Drug_Ind`, `Tot_Srvcs`]. SQLSTATE: 42703", "context": {"file": "line 2 in cell [17]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o870.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Rndrng_Prvdr_Geo_Lvl` cannot be resolved. Did you mean one of the following? [`Rndrng_Prvdr_Geo_Cd`, `Tot_Rndrng_Prvdrs`, `Avg_Sbmtd_Chrg`, `HCPCS_Drug_Ind`, `Tot_Srvcs`]. SQLSTATE: 42703;\n'Filter '`!`('`=`('Rndrng_Prvdr_Geo_Lvl, National))\n+- Relation [Rndrng_Prvdr_

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Rndrng_Prvdr_Geo_Lvl` cannot be resolved. Did you mean one of the following? [`Rndrng_Prvdr_Geo_Cd`, `Tot_Rndrng_Prvdrs`, `Avg_Sbmtd_Chrg`, `HCPCS_Drug_Ind`, `Tot_Srvcs`]. SQLSTATE: 42703;
'Filter '`!`('`=`('Rndrng_Prvdr_Geo_Lvl, National))
+- Relation [Rndrng_Prvdr_Geo_Cd#2941,HCPCS_Cd#2942,HCPCS_Drug_Ind#2943,Place_Of_Srvc#2944,Tot_Rndrng_Prvdrs#2945,Tot_Benes#2946,Tot_Srvcs#2947,Tot_Bene_Day_Srvcs#2948,Avg_Sbmtd_Chrg#2949,Year#2950] csv


In [ ]:
final_df = reduce(DataFrame.unionByName, all_data)
final_df.groupBy("Year").count().orderBy("Year").show()
final_df.show(5)

+----+------+
|Year| count|
+----+------+
|2013|251600|
|2014|251608|
|2015|253084|
|2016|256144|
|2017|256139|
|2018|257249|
|2019|259317|
|2020|254079|
|2021|257325|
|2022|256279|
|2023|254097|
+----+------+

+-------------------+--------+--------------+-------------+-----------------+---------+---------+------------------+--------------+----+
|Rndrng_Prvdr_Geo_Cd|HCPCS_Cd|HCPCS_Drug_Ind|Place_Of_Srvc|Tot_Rndrng_Prvdrs|Tot_Benes|Tot_Srvcs|Tot_Bene_Day_Srvcs|Avg_Sbmtd_Chrg|Year|
+-------------------+--------+--------------+-------------+-----------------+---------+---------+------------------+--------------+----+
|                 01|   00100|             N|            F|              239|      219|    322.0|               322|  1069.9324224|2013|
|                 01|   00103|             N|            F|              515|     2394|   4206.0|              4206|  610.75801474|2013|
|                 01|   00104|             N|            F|              286|      309|   3103.0|       

In [ ]:
final_df.toPandas().to_csv('Data/final_data.csv', index=False)
print("Final cleaned data saved to Data/final_data.csv")

Final cleaned data saved to Data/final_data.csv
